# Train Freshness Classifier
Transfer learning on MobileNetV3-Small, fine-tuned on the grocery freshness dataset.

In [ ]:
import sys
sys.path.append('../src')

import tensorflow as tf
from data_utils import load_datasets, build_augmentation, IMG_SIZE

DATA_DIR = '../data/raw'
BATCH_SIZE = 32
EPOCHS_HEAD = 10
EPOCHS_FINE_TUNE = 10

In [ ]:
train_ds, val_ds, class_names = load_datasets(DATA_DIR, batch_size=BATCH_SIZE)
num_classes = len(class_names)
print(class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## Build model
Freeze the MobileNetV3 backbone first, train only the classification head.

In [ ]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    pooling='avg',
)
base_model.trainable = False

augmentation = build_augmentation()

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
history_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD)

## Fine-tune
Unfreeze the top of the backbone and continue training at a lower learning rate.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE_TUNE)

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f'Validation accuracy: {val_acc:.4f}')

In [ ]:
import pathlib
out_dir = pathlib.Path('../models/saved_model')
out_dir.parent.mkdir(parents=True, exist_ok=True)
model.save(out_dir)
with open('../models/class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))
print('Saved to', out_dir)